# 🚗 Car-Sharing & Urban Mobility — Lab 3

**Politecnico di Torino — ICT for Smart Mobility**  
**Group Project — Individual Contribution**

This portfolio notebook presents the analytical work contributed individually to **Lab 3** of the group project.

### My contribution
- Preliminary vehicle/trip behaviour analysis
- Data filtering and outlier handling
- Vehicle behavioural clustering
- EV model definition and simulation
- AC vs DC charging comparison
- Cluster-level EV performance analysis

> 🔒 **Public portfolio note:** private credentials, university connection details, raw mobility data and local machine paths are intentionally excluded from this public version. Numerical results shown below are derived from the submitted Lab 3 analysis/report.

## 🧭 Analysis Roadmap

1. 🔎 Preliminary Vehicle Behaviour Analysis
2. 🧹 Data Filtering & Outlier Removal
3. 🧩 Vehicle Behaviour Clustering
4. 🔋 EV Models & Simulation Setup
5. ⚡ EV Simulation Results — AC vs DC
6. 📊 EV Performance Metrics
7. 🧠 Cluster-Level EV Performance
8. 🎯 Key Takeaways

## 1. 🔎 Preliminary Vehicle Behaviour Analysis

The analysis examined trip distance and duration across road types:

- **U — Urban**
- **E — Extra-Urban**
- **A — Highway**
- Unknown/other road types

The submitted report shows that urban trips are concentrated at shorter distances, while highway trips are more associated with longer distances. The raw distance-duration relationship also contained substantial variability and potential outliers.

In [ ]:
import pandas as pd

preliminary_summary = pd.DataFrame({
    "Road type": ["Urban (U)", "Extra-Urban (E)", "Highway (A)", "Unknown"],
    "Observed pattern": [
        "Predominantly shorter trips",
        "Intermediate distance/duration patterns",
        "Predominantly longer trips",
        "Intermediate / mixed patterns"
    ]
})
preliminary_summary

      Road type                          Observed pattern
0    Urban (U)            Predominantly shorter trips
1    Extra-Urban (E)     Intermediate distance/duration patterns
2    Highway (A)         Predominantly longer trips
3    Unknown              Intermediate / mixed patterns

### Raw trip distance-duration relationship

The submitted report calculated a **Pearson correlation of r = 0.025 before filtering**. The report interprets this as evidence that the raw data contains substantial noise and outliers, motivating the filtering stage.

In [ ]:
distance_duration_raw = pd.DataFrame({
    "Dataset stage": ["Before filtering"],
    "Pearson r (distance vs duration)": [0.025]
})
distance_duration_raw

  Dataset stage  Pearson r (distance vs duration)
0 Before filtering                            0.025

## 2. 🧹 Data Filtering & Outlier Removal

The submitted analysis applied the following filters:

- Remove records with **distance = 0 km**
- Remove records with **duration ≤ 0 seconds**
- Remove distances **> 3000 km**
- Remove velocities **> 160 km/h**
- Remove velocities **< 0.5 km/h**
- Remove records with **distance > 100 km and duration < 60 seconds**

The report explicitly notes that these counts refer to **dataset rows/records, not unique trips**, because one trip can contain multiple road-type segments.

In [ ]:
filtering_summary = pd.DataFrame({
    "Stage": ["Before filtering", "After filtering"],
    "Rows / records": [1415305, 1247373]
})
filtering_summary["Rows removed"] = [0, 1415305 - 1247373]
filtering_summary

               Stage  Rows / records  Rows removed
0   Before filtering         1415305             0
1    After filtering         1247373        167932

## 3. 🧩 Vehicle Behaviour Clustering

Vehicles were grouped according to their driving/road-usage patterns. The final analysis produced **four behavioural clusters**.

The source report states that a 3D projection was used only for visualisation and did not change the clustering logic.

In [ ]:
cluster_summary = pd.DataFrame({
    "Cluster": [0, 1, 2, 3],
    "Number of vehicles": [390, 521, 88, 1],
    "Behaviour summary": [
        "Balanced usage across road types (A/E/U)",
        "Urban-focused vehicles (mostly U roads)",
        "High Extra-Urban/Highway usage (E & A)",
        "Very low usage / Outlier"
    ]
})
cluster_summary

   Cluster  Number of vehicles                         Behaviour summary
0        0                 390       Balanced usage across road types (A/E/U)
1        1                 521       Urban-focused vehicles (mostly U roads)
2        2                  88       High Extra-Urban/Highway usage (E & A)
3        3                   1       Very low usage / Outlier

## 4. 🔋 EV Models & Simulation Setup

Three EV models were selected with different battery capacities, consumption rates, charging powers and ranges.

In [ ]:
ev_specs = pd.DataFrame({
    "Model": ["Hyundai Kona Electric", "Tesla Model 3 Long Range", "Nissan Leaf"],
    "Battery capacity (kWh)": [64, 75, 40],
    "Energy consumption (Wh/km)": [150, 170, 150],
    "AC charging power (kW)": [7.2, 11.5, 6.6],
    "DC charging power (kW)": [100, 250, 50],
    "Range (km)": [480, 585, 241]
})
ev_specs

                     Model  Battery capacity (kWh)  Energy consumption (Wh/km)  AC charging power (kW)  DC charging power (kW)  Range (km)
0     Hyundai Kona Electric                       64                          150                       7.2                     100         480
1   Tesla Model 3 Long Range                       75                          170                      11.5                     250         585
2              Nissan Leaf                       40                          150                       6.6                      50         241

### Simulation logic

For each vehicle, trips are ordered chronologically. The simulator:

1. Starts with **SoC = 100%**
2. Calculates trip energy: `E_trip = distance × consumption / 1000`
3. Checks whether available SoC is sufficient
4. Updates SoC after the trip
5. Uses the parking interval before the next trip as a charging opportunity
6. Recharges when parking time is **≥ 20 minutes**
7. Caps SoC at 100%
8. Stores feasibility, SoC, energy and cost metrics

The original analysis simulated **1,048,576 trips**.

In [ ]:
def trip_energy_kwh(distance_km, consumption_wh_per_km):
    return distance_km * consumption_wh_per_km / 1000

def charging_cost(energy_kwh, tariff_eur_per_kwh):
    return energy_kwh * tariff_eur_per_kwh

example = pd.DataFrame({
    "Example distance (km)": [10, 25],
    "Kona energy (kWh)": [
        trip_energy_kwh(10, 150),
        trip_energy_kwh(25, 150)
    ]
})
example

   Example distance (km)  Kona energy (kWh)
0                      10                1.50
1                      25                3.75

## 5. ⚡ EV Simulation Results — AC vs DC

The **Group13-LAB3_V2 report** is used as the main source for the final comparison tables below.

### Infeasible trips

In [ ]:
infeasible = pd.DataFrame({
    "Model": ["Nissan Leaf", "Tesla Model 3", "Hyundai Kona"],
    "AC infeasible (%)": [13.96, 4.67, 6.31],
    "DC infeasible (%)": [7.27, 1.85, 2.08]
})
infeasible

            Model  AC infeasible (%)  DC infeasible (%)
0    Nissan Leaf              13.96               7.27
1   Tesla Model 3               4.67               1.85
2  Hyundai Kona                 6.31               2.08

### Feasible trips

In [ ]:
feasible = pd.DataFrame({
    "Model": ["Nissan Leaf", "Tesla Model 3", "Hyundai Kona"],
    "AC feasible (%)": [86.04, 95.33, 93.69],
    "DC feasible (%)": [92.73, 98.15, 97.92]
})
feasible

            Model  AC feasible (%)  DC feasible (%)
0    Nissan Leaf            86.04             92.73
1   Tesla Model 3            95.33             98.15
2  Hyundai Kona              93.69             97.92

## 6. 📊 EV Performance Metrics

The report compares average energy consumption and charging cost under AC and DC charging.

In [ ]:
performance = pd.DataFrame({
    "Model": ["Nissan Leaf", "Tesla Model 3", "Hyundai Kona"],
    "Energy/km AC (kWh/km)": [0.46, 0.60, 0.51],
    "Energy/km DC (kWh/km)": [0.51, 0.64, 0.56],
    "Cost/trip AC (€)": [0.37, 0.53, 0.45],
    "Cost/trip DC (€)": [0.70, 0.97, 0.85]
})
performance

            Model  Energy/km AC (kWh/km)  Energy/km DC (kWh/km)  Cost/trip AC (€)  Cost/trip DC (€)
0    Nissan Leaf                    0.46                   0.51               0.37               0.70
1   Tesla Model 3                    0.60                   0.64               0.53               0.97
2  Hyundai Kona                     0.51                   0.56               0.45               0.85

### SoC patterns reported

The submitted report describes the observed SoC distributions as follows:

- **Tesla Model 3:** commonly above 70% after feasible trips.
- **Hyundai Kona:** concentrated approximately in the 55–65% range.
- **Nissan Leaf:** lower distribution, with many values around 30–40%.

These observations come from the report's AC/DC SoC plots.

## 7. 🧠 Cluster-Level EV Performance — DC Charging

The three EV models were evaluated under the four behavioural clusters obtained from the vehicle clustering stage.

### Cluster 0 — Mixed driving pattern

In [ ]:
cluster0 = pd.DataFrame({
    "Metric": ["Feasibility (%)", "SoC after trip (%)", "Energy/km", "Cost/trip (€)"],
    "Tesla Model 3": [91.55, 61.20, 0.462, 3.751],
    "Nissan Leaf": [76.47, 28.32, 0.405, 3.309],
    "Hyundai Kona": [90.68, 51.29, 0.408, 3.309]
})
cluster0

                Metric  Tesla Model 3  Nissan Leaf  Hyundai Kona
0       Feasibility (%)          91.550       76.470        90.680
1  SoC after trip (%)          61.200       28.320        51.290
2            Energy/km           0.462        0.405         0.408
3         Cost/trip (€)           3.751        3.309         3.309

### Cluster 1 — Short, frequent urban trips

In [ ]:
cluster1 = pd.DataFrame({
    "Metric": ["Feasibility (%)", "SoC after trip (%)", "Energy/km", "Cost/trip (€)"],
    "Tesla Model 3": [99.68, 65.32, 0.394, 0.756],
    "Nissan Leaf": [97.05, 31.39, 0.342, 0.667],
    "Hyundai Kona": [99.62, 55.43, 0.347, 0.667]
})
cluster1

                Metric  Tesla Model 3  Nissan Leaf  Hyundai Kona
0       Feasibility (%)          99.680       97.050        99.620
1  SoC after trip (%)          65.320       31.390        55.430
2            Energy/km           0.394        0.342         0.347
3         Cost/trip (€)           0.756        0.667         0.667

### Cluster 2 — Long-distance travel

In [ ]:
cluster2 = pd.DataFrame({
    "Metric": ["Feasibility (%)", "SoC after trip (%)", "Energy/km", "Cost/trip (€)"],
    "Tesla Model 3": [97.83, 62.48, 0.450, 1.700],
    "Nissan Leaf": [90.12, 29.26, 0.391, 1.500],
    "Hyundai Kona": [97.52, 52.84, 0.397, 1.500]
})
cluster2

                Metric  Tesla Model 3  Nissan Leaf  Hyundai Kona
0       Feasibility (%)          97.830       90.120        97.520
1  SoC after trip (%)          62.480       29.260        52.840
2            Energy/km           0.450        0.391         0.397
3         Cost/trip (€)           1.700        1.500         1.500

### Cluster 3 — Outlier / rare-use behaviour

The report describes this cluster as containing only **62 trips** and as non-representative. Its reported DC metrics are:

In [ ]:
cluster3 = pd.DataFrame({
    "Metric": ["Feasibility (%)", "SoC after trip (%)", "Energy/km", "Cost/trip (€)"],
    "Tesla Model 3": [9.09, 14.27, 0.340, 92.102],
    "Nissan Leaf": [9.09, 5.42, 0.300, 81.266],
    "Hyundai Kona": [9.09, 12.10, 0.300, 81.266]
})
cluster3

                Metric  Tesla Model 3  Nissan Leaf  Hyundai Kona
0       Feasibility (%)           9.090        9.090         9.090
1  SoC after trip (%)          14.270        5.420        12.100
2            Energy/km           0.340        0.300         0.300
3         Cost/trip (€)          92.102       81.266        81.266

> **Source note:** The report prints the Tesla Cluster 3 cost as `92..102`; the numeric value above is recorded as **92.102** because that is the apparent intended value. Cluster 3 is very small and the report recommends cautious interpretation.

## 8. 🎯 Key Takeaways

Based on the submitted Lab 3 analysis:

- The raw mobility data contained substantial noise/outliers, motivating explicit distance, duration and velocity filtering.
- Four behavioural vehicle clusters were identified from road-usage patterns.
- The EV simulation evaluated **1,048,576 trips** under AC and DC charging scenarios.
- In the final comparison reported in **Group13-LAB3_V2**, DC charging reduced infeasible trips for all three EV models.
- The effect of charging mode was particularly visible for the Nissan Leaf.
- Cluster-level results show that vehicle behaviour affects EV feasibility, remaining SoC and charging cost.
- Cluster 3 is a very small, non-representative group and should be interpreted cautiously.

## 🧰 Tools & Methods

**Python · Pandas · NumPy · Matplotlib · K-Means · EV simulation · Data filtering · Behavioural clustering · Data visualization**

### Source note

This public portfolio version is based on the submitted **Group 13 Lab 3 report and analysis**. The original raw mobility data, private credentials and local machine paths are not included.